# 🏔️ Semana 12 · Unidad 3 — Laboratorio: Heapsort

**Universidad de Talca — Curso de Algoritmos y Estructuras de Datos**

| Aspecto | Detalle |
|--------|--------|
| **Profesor** | PhD. César Astudillo |
| **Unidad** | Unidad 3: Algoritmos de ordenamiento |
| **Tema** | Heapsort — heapify, sortdown y cierre comparativo de la unidad |
| **Duración** | 100 minutos (2 bloques de 50 min) |

---

## 📋 Instrucciones Generales

- Trabaja de forma **individual**.
- Ejecuta cada celda antes de pasar a la siguiente.
- El verificador automático te dirá cuántos casos pasan. Apunta a 100%.
- Al terminar el Bloque 1, el ayudante revisará tu avance antes de continuar.
- **No modifiques** las celdas de Setup ni las de verificación.

> 📌 **Prerrequisito.** Este laboratorio asume que ya implementaste `swim` y `sink` en el
> laboratorio de la Semana 11 (Colas de Prioridad). Aquí los reutilizamos para ordenar
> **in-place**, sin estructura auxiliar.

> ⚠️ **Entrega de la Tarea de la Unidad 3** al final de esta sesión.

In [ ]:
# Setup — ejecutar primero. No modificar.
import sys, time, random, math
import numpy as np
import matplotlib.pyplot as plt

random.seed(2026)
np.random.seed(2026)

for nombre in ("numpy", "matplotlib"):
    try:
        __import__(nombre)
        print(f"✅ {nombre} disponible")
    except ImportError:
        print(f"❌ {nombre} NO encontrado — instala con: pip install {nombre}")

print("\n🐍 Python", sys.version.split()[0], "| Listo para comenzar.")

# 🔵 BLOQUE 1 — Fase 1: Heapify (50 minutos)

Heapsort tiene dos fases. La primera convierte un arreglo cualquiera en un max-heap
**usando el mismo arreglo**, sin memoria extra. Esa es la fase que trabajamos aquí.

> 💡 **Insight:** un heap no es una estructura nueva en memoria. Es una *forma* que le
> imponemos al arreglo que ya tenemos. Por eso Heapsort puede ordenar in-place.

## PARTE 1A: Trazar heapify a mano (10 minutos)

Usamos indexación **base 0**: para el nodo `i`, sus hijos son `2i+1` y `2i+2`.

Considera el arreglo:

```
índice:  0   1   2   3   4   5
valor:  [3,  7,  1,  9,  4,  8]
```

Como árbol:

```
              3(0)
            /      \
        7(1)        1(2)
       /    \      /
    9(3)   4(4)  8(5)
```

`heapify` bottom-up recorre los nodos internos **desde el último hacia el primero** y
aplica `sink` a cada uno. Con n=6, el último nodo interno es el índice `n//2 - 1 = 2`.

### Pregunta 1 — orden de visita
¿Sobre qué índices se llama `sink`, y en qué orden?

### Pregunta 2 — traza
Escribe el estado del arreglo después de cada llamada a `sink`.

### Pregunta 3 — ¿por qué al revés?
¿Qué pasaría si recorriéramos los nodos de arriba hacia abajo (índice 0 hacia n)?
Da un contraejemplo concreto con este mismo arreglo.

> 🎙️ **[PAUSA PROFESOR]** Pregunta sugerida: "¿Por qué no hace falta llamar a `sink`
> sobre las hojas?"

### Tu Respuesta 1A (edita esta celda)

**P1 — orden de visita:**

**P2 — traza:**

| Llamada | Arreglo resultante |
|---|---|
| inicial | `[3, 7, 1, 9, 4, 8]` |
| `sink(2)` | |
| `sink(1)` | |
| `sink(0)` | |

**P3 — por qué bottom-up:**

## PARTE 1B: Implementar sink (12 minutos)

`sink(a, i, n)` hace descender el elemento en la posición `i` hasta que la propiedad de
max-heap se restablezca, considerando solo las primeras `n` posiciones del arreglo.

El parámetro `n` es lo que más adelante nos permitirá "congelar" la cola ordenada del
arreglo: en la fase 2 el heap se irá encogiendo mientras el arreglo completo no cambia
de tamaño.

In [ ]:
def sink(a: list, i: int, n: int, stats: dict | None = None) -> None:
    """
    Hunde el elemento a[i] hasta restaurar la propiedad de max-heap en a[0:n].

    Parámetros:
        a (list): arreglo que representa el heap (indexación base 0)
        i (int): índice del elemento a hundir
        n (int): tamaño efectivo del heap (solo se considera a[0:n])
        stats (dict|None): si se entrega, acumula 'comparaciones' e 'intercambios'

    Retorna:
        None — modifica el arreglo in-place.

    Complejidad:
        Temporal: O(log n) — a lo más recorre la altura del heap
        Espacial: O(1) — iterativo, sin recursión
    """
    # Tu código aquí.
    # Pista: mientras el hijo izquierdo (2*i+1) esté dentro de n:
    #   1. elige el hijo MAYOR entre 2*i+1 y 2*i+2 (si el derecho existe)
    #   2. si a[i] ya es >= que ese hijo, termina
    #   3. si no, intercambia y continúa desde la posición del hijo
    # Si stats no es None, incrementa stats['comparaciones'] y stats['intercambios'].
    pass

In [ ]:
def verificar_sink(fn):
    """Verifica sink sobre casos construidos a mano."""
    def es_max_heap(a, n):
        for i in range(n):
            for h in (2*i+1, 2*i+2):
                if h < n and a[h] > a[i]:
                    return False
        return True

    # OJO: sink SUPONE que los subárboles hijos ya son heaps; solo la posición i
    # puede violar la propiedad. Todos los casos de abajo respetan esa precondición.
    casos = [
        ([3, 9, 8], 0, 3, "raíz baja, dos hijos"),
        ([1, 5, 4, 3, 2], 0, 5, "raíz mínima, hunde dos niveles"),
        ([9, 5, 4, 1, 2], 0, 5, "ya es heap, no debe cambiar"),
        ([5], 0, 1, "un solo elemento"),
        ([1, 7], 0, 2, "solo hijo izquierdo"),
        ([10, 1, 9, 8, 7, 6, 5], 1, 7, "hundir un nodo interno, no la raíz"),
    ]
    aprobados = 0
    for arr, i, n, desc in casos:
        a = list(arr)
        try:
            fn(a, i, n)
            mismo_multiconjunto = sorted(a) == sorted(arr)
            if es_max_heap(a, n) and mismo_multiconjunto:
                print(f"  ✅ {desc} → {a}")
                aprobados += 1
            else:
                print(f"  ❌ {desc}")
                print(f"     Entrada:  {arr} (i={i}, n={n})")
                print(f"     Obtenido: {a}")
        except Exception as e:
            print(f"  💥 {desc} — Error: {e}")
    print(f"\n{'🎉 Todos los casos pasaron!' if aprobados == len(casos) else f'⚠️  {aprobados}/{len(casos)} casos correctos'}")

verificar_sink(sink)

## PARTE 1C: Implementar heapify (13 minutos)

Ahora construye el heap completo. La versión ingenua sería insertar los `n` elementos uno
por uno con `swim`, lo que cuesta $O(n \log n)$. La versión bottom-up con `sink` cuesta
$O(n)$ — y la vas a medir en la parte 1D.

In [ ]:
def heapify(a: list, stats: dict | None = None) -> None:
    """
    Convierte el arreglo a en un max-heap in-place, recorriendo bottom-up.

    Parámetros:
        a (list): arreglo a transformar
        stats (dict|None): acumulador opcional de operaciones

    Retorna:
        None — modifica el arreglo in-place.

    Complejidad:
        Temporal: O(n) — la suma sum_{h} (n/2^{h+1}) * h converge a n
        Espacial: O(1)
    """
    # Tu código aquí.
    # Recorre los nodos internos desde n//2 - 1 hasta 0 (inclusive, hacia atrás)
    # y llama a sink sobre cada uno.
    pass


def heapify_ingenuo(a: list) -> list:
    """
    Versión de contraste: construye el heap insertando uno por uno con swim.

    Complejidad:
        Temporal: O(n log n)
        Espacial: O(n) — construye un arreglo nuevo
    """
    heap = []
    for x in a:
        heap.append(x)
        j = len(heap) - 1
        while j > 0 and heap[(j - 1) // 2] < heap[j]:
            heap[(j - 1) // 2], heap[j] = heap[j], heap[(j - 1) // 2]
            j = (j - 1) // 2
    return heap

In [ ]:
def verificar_heapify(fn):
    """Verifica que heapify produzca un max-heap válido sin perder elementos."""
    def es_max_heap(a):
        n = len(a)
        return all(a[h] <= a[i] for i in range(n) for h in (2*i+1, 2*i+2) if h < n)

    casos = [
        ([3, 7, 1, 9, 4, 8], "el ejemplo de la parte 1A"),
        ([], "arreglo vacío"),
        ([42], "un elemento"),
        ([1, 2, 3, 4, 5, 6, 7, 8], "orden creciente (peor forma inicial)"),
        ([8, 7, 6, 5, 4, 3, 2, 1], "orden decreciente (ya es heap)"),
        ([5, 5, 5, 5], "todos iguales"),
        ([random.randint(0, 100) for _ in range(50)], "aleatorio n=50"),
    ]
    aprobados = 0
    for arr, desc in casos:
        a = list(arr)
        try:
            fn(a)
            if es_max_heap(a) and sorted(a) == sorted(arr):
                print(f"  ✅ {desc}")
                aprobados += 1
            else:
                print(f"  ❌ {desc} — obtenido: {a[:12]}{'...' if len(a) > 12 else ''}")
        except Exception as e:
            print(f"  💥 {desc} — Error: {e}")
    print(f"\n{'🎉 Todos los casos pasaron!' if aprobados == len(casos) else f'⚠️  {aprobados}/{len(casos)} casos correctos'}")

verificar_heapify(heapify)

## PARTE 1D: Comprobar que heapify es O(n) (15 minutos)

Aquí está el punto pedagógico del bloque. La intuición dice que construir un heap de `n`
elementos debería costar $O(n \log n)$: son `n` elementos y cada `sink` cuesta $O(\log n)$.

Pero esa cota es **floja**. La mayoría de los nodos están cerca de las hojas y casi no se
hunden. La suma real es:

$$\sum_{h=0}^{\lfloor \log n \rfloor} \frac{n}{2^{h+1}} \cdot h \;=\; O(n)$$

Mide el número de intercambios y verifica que crece **linealmente**.

In [ ]:
# Medición: intercambios de heapify vs n
tamanos = [1000, 2000, 4000, 8000, 16000, 32000]
intercambios = []

for n in tamanos:
    datos = [random.randint(0, 10**6) for _ in range(n)]
    stats = {"comparaciones": 0, "intercambios": 0}
    heapify(datos, stats)
    intercambios.append(stats["intercambios"])

print(f"{'n':>8} {'intercambios':>14} {'intercambios/n':>16}")
print("-" * 40)
for n, s in zip(tamanos, intercambios):
    print(f"{n:>8} {s:>14} {s/n:>16.3f}")

print("\n👉 Si la última columna se mantiene aproximadamente CONSTANTE,")
print("   entonces intercambios ≈ c·n, es decir heapify es O(n).")

In [ ]:
# Gráfico: heapify O(n) frente a la cota floja O(n log n)
fig, ax = plt.subplots(figsize=(10, 6))
n_arr = np.array(tamanos, dtype=float)
s_arr = np.array(intercambios, dtype=float)

ax.plot(n_arr, s_arr, "o-", linewidth=2, label="heapify bottom-up (medido)")
if s_arr[0] > 0:
    ax.plot(n_arr, s_arr[0] * (n_arr / n_arr[0]), "--", linewidth=2,
            label="referencia O(n)")
    ax.plot(n_arr, s_arr[0] * (n_arr * np.log2(n_arr)) / (n_arr[0] * np.log2(n_arr[0])),
            ":", linewidth=2, label="referencia O(n log n)")

ax.set_xlabel("Tamaño de entrada (n)")
ax.set_ylabel("Intercambios")
ax.set_title("heapify bottom-up crece linealmente, no en n log n")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### ✋ Punto de control del Bloque 1

Muestra al ayudante:
1. Tu traza de la Parte 1A.
2. `verificar_sink` y `verificar_heapify` en 🎉.
3. El gráfico de la Parte 1D, y explica en una frase por qué la curva medida sigue a la
   recta $O(n)$ y no a la curva $O(n\log n)$.

# 🟠 BLOQUE 2 — Fase 2: Sortdown y cierre de la unidad (50 minutos)

Con el max-heap construido, el máximo está en `a[0]`. La fase 2 lo lleva a su posición
final y encoge el heap en uno. Repetido `n-1` veces, el arreglo queda ordenado.

## PARTE 2A: Trazar sortdown a mano (10 minutos)

Partimos del max-heap `[9, 7, 8, 3, 4, 1]` (n=6).

El paso de sortdown es siempre el mismo:
1. Intercambia `a[0]` con `a[n-1]`.
2. Reduce el tamaño efectivo del heap: `n ← n-1`.
3. Llama a `sink(a, 0, n)`.

### Pregunta 1 — dos iteraciones
Escribe el arreglo completo (incluida la zona ya ordenada) tras las dos primeras iteraciones.
Marca con `|` la frontera entre heap y zona ordenada.

### Pregunta 2 — ¿por qué queda ordenado ascendente?
El heap es de **máximos** y el resultado queda de menor a mayor. Explica por qué.

### Pregunta 3 — estabilidad
Ordena a mano `[(2,'a'), (1,'x'), (2,'b')]` por la primera componente con Heapsort.
¿Se conserva el orden relativo de `(2,'a')` y `(2,'b')`? ¿Qué dice eso de Heapsort?

> 🎙️ **[PAUSA PROFESOR]** Pregunta sugerida: "¿Cuántos elementos quedan en el heap
> después de k iteraciones de sortdown?"

### Tu Respuesta 2A (edita esta celda)

**P1 — traza:**

| Iteración | Arreglo (heap `|` ordenado) |
|---|---|
| inicial | `[9, 7, 8, 3, 4, 1]` `|` — |
| 1 | |
| 2 | |

**P2 — por qué ascendente:**

**P3 — estabilidad:**

## PARTE 2B: Implementar Heapsort (18 minutos)

In [ ]:
def heapsort(a: list, stats: dict | None = None) -> list:
    """
    Ordena el arreglo a de menor a mayor, in-place, usando Heapsort.

    Parámetros:
        a (list): arreglo a ordenar
        stats (dict|None): acumulador opcional de 'comparaciones' e 'intercambios'

    Retorna:
        list: el mismo arreglo a, ya ordenado (se retorna por comodidad)

    Complejidad:
        Temporal: O(n log n) en TODOS los casos — heapify O(n) + n·sink O(log n)
        Espacial: O(1) — in-place, sin arreglos auxiliares

    Ejemplo:
        >>> heapsort([3, 1, 2])
        [1, 2, 3]
    """
    # Tu código aquí.
    # Fase 1: heapify(a, stats)
    # Fase 2: para n desde len(a) hasta 2:
    #           intercambia a[0] con a[n-1]
    #           sink(a, 0, n-1, stats)
    pass

In [ ]:
def verificar_heapsort(fn):
    """Verifica correctitud de heapsort, incluido que ordene in-place."""
    casos = [
        ([3, 1, 2], "caso mínimo"),
        ([], "arreglo vacío"),
        ([7], "un elemento"),
        ([1, 2, 3, 4, 5, 6], "ya ordenado"),
        ([6, 5, 4, 3, 2, 1], "orden inverso"),
        ([4, 4, 4, 4], "todos iguales"),
        ([-3, 10, -7, 0, 2, -7], "con negativos y repetidos"),
        ([random.randint(-50, 50) for _ in range(200)], "aleatorio n=200"),
    ]
    aprobados = 0
    for arr, desc in casos:
        a = list(arr)
        try:
            ret = fn(a)
            esperado = sorted(arr)
            ok_valor = a == esperado
            ok_inplace = ret is None or ret is a
            if ok_valor and ok_inplace:
                print(f"  ✅ {desc}")
                aprobados += 1
            elif not ok_inplace:
                print(f"  ❌ {desc} — debe ordenar IN-PLACE (retorna el mismo objeto o None)")
            else:
                print(f"  ❌ {desc}")
                print(f"     Esperado: {esperado[:12]}{'...' if len(esperado) > 12 else ''}")
                print(f"     Obtenido: {a[:12]}{'...' if len(a) > 12 else ''}")
        except Exception as e:
            print(f"  💥 {desc} — Error: {e}")
    print(f"\n{'🎉 Todos los casos pasaron!' if aprobados == len(casos) else f'⚠️  {aprobados}/{len(casos)} casos correctos'}")

verificar_heapsort(heapsort)

## PARTE 2C: Heapsort no es estable — demuéstralo (7 minutos)

Merge Sort es estable; Heapsort no. En vez de creerlo, constrúyelo: encuentra una entrada
concreta donde Heapsort altera el orden relativo de dos elementos con la misma clave.

> ⚠️ **Importante:** la estabilidad importa cuando ordenas por varios criterios en cadena.
> Si ordenas por nombre y luego por nota con un algoritmo inestable, pierdes el orden por
> nombre dentro de cada nota.

In [ ]:
# Buscamos un contraejemplo de estabilidad de forma automática.
# Cada elemento es (clave, marca_de_origen). Ordenamos SOLO por clave.

def heapsort_por_clave(pares):
    """Aplica tu heapsort usando únicamente la primera componente como clave."""
    # Truco: ordenamos una lista de índices decorados para no depender
    # de cómo tu heapsort compare tuplas.
    a = [(k, i, m) for i, (k, m) in enumerate(pares)]
    # Comparamos solo por k: reemplazamos el arreglo por claves y reordenamos en paralelo.
    # Para simplificar, usamos tu sink/heapify sobre una copia con clave única artificial.
    trabajo = [(k, m) for (k, i, m) in a]
    # Implementación directa de heapsort comparando solo trabajo[x][0]:
    def sink_k(arr, i, n):
        while 2*i + 1 < n:
            j = 2*i + 1
            if j + 1 < n and arr[j+1][0] > arr[j][0]:
                j += 1
            if arr[i][0] >= arr[j][0]:
                break
            arr[i], arr[j] = arr[j], arr[i]
            i = j
    n = len(trabajo)
    for i in range(n//2 - 1, -1, -1):
        sink_k(trabajo, i, n)
    for m in range(n, 1, -1):
        trabajo[0], trabajo[m-1] = trabajo[m-1], trabajo[0]
        sink_k(trabajo, 0, m-1)
    return trabajo


encontrado = None
for intento in range(3000):
    n = random.randint(3, 7)
    # pocas claves distintas => muchos empates
    pares = [(random.randint(0, 2), f"e{i}") for i in range(n)]
    salida = heapsort_por_clave(pares)
    estable = sorted(pares, key=lambda p: p[0])
    if salida != estable:
        encontrado = (pares, salida, estable)
        break

if encontrado:
    entrada, obtenido, estable = encontrado
    print("Contraejemplo de estabilidad encontrado:\n")
    print(f"  Entrada          : {entrada}")
    print(f"  Heapsort devuelve: {obtenido}")
    print(f"  Un orden estable : {estable}")
    print("\n👉 Los elementos con la misma clave cambiaron de orden relativo.")
    print("   Heapsort NO es estable.")
else:
    print("No se halló contraejemplo en 3000 intentos (sube el número de intentos).")

## PARTE 2D: El cierre de la Unidad 3 (15 minutos)

Esta es la última sesión de contenidos de la unidad. Compara **los seis algoritmos** que
estudiaste sobre los mismos datos y sobre distintos perfiles de entrada.

> 🎙️ **[PAUSA PROFESOR]** Pregunta sugerida: "Si Heapsort es $O(n\log n)$ garantizado
> y ordena in-place, ¿por qué `sorted()` de Python no lo usa?\"

In [ ]:
# Setup de comparación — implementaciones de referencia de la unidad. No modificar.
def selection_sort(a):
    a = list(a)
    for i in range(len(a)):
        m = i
        for j in range(i+1, len(a)):
            if a[j] < a[m]:
                m = j
        a[i], a[m] = a[m], a[i]
    return a

def insertion_sort(a):
    a = list(a)
    for i in range(1, len(a)):
        v, j = a[i], i - 1
        while j >= 0 and a[j] > v:
            a[j+1] = a[j]
            j -= 1
        a[j+1] = v
    return a

def shell_sort(a):
    a = list(a)
    n, h = len(a), 1
    while h < n // 3:
        h = 3*h + 1
    while h >= 1:
        for i in range(h, n):
            v, j = a[i], i
            while j >= h and a[j-h] > v:
                a[j] = a[j-h]
                j -= h
            a[j] = v
        h //= 3
    return a

def merge_sort(a):
    if len(a) <= 1:
        return list(a)
    m = len(a) // 2
    iz, de = merge_sort(a[:m]), merge_sort(a[m:])
    out, i, j = [], 0, 0
    while i < len(iz) and j < len(de):
        if iz[i] <= de[j]:
            out.append(iz[i]); i += 1
        else:
            out.append(de[j]); j += 1
    out.extend(iz[i:]); out.extend(de[j:])
    return out

def quicksort(a):
    a = list(a)
    def qs(lo, hi):
        while lo < hi:
            p = random.randint(lo, hi)
            a[p], a[hi] = a[hi], a[p]
            piv, i = a[hi], lo
            for j in range(lo, hi):
                if a[j] < piv:
                    a[i], a[j] = a[j], a[i]
                    i += 1
            a[i], a[hi] = a[hi], a[i]
            if i - lo < hi - i:
                qs(lo, i-1); lo = i + 1
            else:
                qs(i+1, hi); hi = i - 1
    qs(0, len(a)-1)
    return a

def heapsort_copia(a):
    b = list(a)
    heapsort(b)
    return b

ALGORITMOS = {
    "Selection": selection_sort,
    "Insertion": insertion_sort,
    "Shell":     shell_sort,
    "Merge":     merge_sort,
    "Quicksort": quicksort,
    "Heapsort":  heapsort_copia,
}
print("✅ Algoritmos de referencia cargados:", ", ".join(ALGORITMOS))

In [ ]:
# Comparación sobre tres perfiles de entrada
N = 2000
perfiles = {
    "Aleatorio":       [random.randint(0, 10**6) for _ in range(N)],
    "Ya ordenado":     list(range(N)),
    "Orden inverso":   list(range(N, 0, -1)),
    "Casi ordenado":   None,
}
casi = list(range(N))
for _ in range(N // 50):
    i, j = random.randrange(N), random.randrange(N)
    casi[i], casi[j] = casi[j], casi[i]
perfiles["Casi ordenado"] = casi

sys.setrecursionlimit(10000)
resultados = {}
print(f"Tiempos en milisegundos (n = {N})\n")
print(f"{'Algoritmo':<12}" + "".join(f"{p:>16}" for p in perfiles))
print("-" * (12 + 16*len(perfiles)))
for nombre, fn in ALGORITMOS.items():
    fila = []
    for datos in perfiles.values():
        t0 = time.perf_counter()
        salida = fn(datos)
        t1 = time.perf_counter()
        assert salida == sorted(datos), f"{nombre} produjo un resultado incorrecto"
        fila.append((t1 - t0) * 1000)
    resultados[nombre] = fila
    print(f"{nombre:<12}" + "".join(f"{t:>16.1f}" for t in fila))

In [ ]:
# Gráfico comparativo
fig, ax = plt.subplots(figsize=(11, 6))
etiquetas = list(perfiles.keys())
x = np.arange(len(etiquetas))
ancho = 0.13

for k, (nombre, fila) in enumerate(resultados.items()):
    ax.bar(x + k*ancho, fila, ancho, label=nombre)

ax.set_xticks(x + ancho * (len(resultados)-1) / 2)
ax.set_xticklabels(etiquetas)
ax.set_ylabel("Tiempo (ms, escala log)")
ax.set_yscale("log")
ax.set_title(f"Los seis algoritmos de la Unidad 3 sobre distintos perfiles (n={N})")
ax.legend(ncol=3)
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

### Preguntas de Análisis (edita esta celda)

1. **Insertion Sort** es el más lento en entrada aleatoria pero de los más rápidos en
   "casi ordenado". ¿Por qué? ¿Qué propiedad tiene que los demás no?

2. **Heapsort** no gana en ninguna columna, pero tampoco se derrumba en ninguna. Describe
   en qué escenario real preferirías esa garantía por sobre la velocidad promedio de
   Quicksort.

3. Compara **Merge** y **Heapsort**: ambos son $O(n\log n)$ garantizado. ¿Qué gana y qué
   pierde cada uno? Menciona memoria y estabilidad.

4. `sorted()` de Python usa **Timsort**, un híbrido de Merge e Insertion. A la luz de la
   columna "Casi ordenado", ¿por qué es una buena decisión de diseño?

**Tus respuestas:**

## 🔬 Zona de Experimentación

Las siguientes celdas son tuyas. Algunas sugerencias:
- ¿Qué pasa si cambias `N` a 20000? ¿Qué algoritmos se vuelven impracticables?
- Adapta `heapsort` para producir orden **descendente**. ¿Basta con invertir al final,
  o conviene usar un min-heap?
- Mide el número de comparaciones (usando `stats`) en lugar del tiempo. ¿Cambia el ranking?

In [ ]:
# Espacio libre para experimentar
# Sugerencia: repite la comparación con N = 20000 quitando Selection e Insertion.


In [ ]:
# Espacio libre para experimentar
# Sugerencia: implementa heapsort_descendente usando un min-heap.


## 📤 Entrega de la Tarea de la Unidad 3

Al cierre de esta sesión se entrega la **Tarea de la Unidad 3** (producto computacional,
10% del Área N°3).

- Enunciado: [`S10_ALG_ASIG_Quicksort.ipynb`](../S10_U3_Quicksort/S10_ALG_ASIG_Quicksort.ipynb)
- Formato: notebook ejecutado + informe breve, según indica el enunciado.

> ⚠️ La **Prueba de la Unidad 3** cubre desde ordenamiento elemental hasta Heapsort
> (todo lo trabajado entre las semanas 7 y 12). Symbol Tables y BST **no** entran.

## 📚 Lecturas Recomendadas y Práctica

### Textbooks

| Libro | Edición | Capítulo | Tema |
|-------|---------|----------|------|
| Cormen et al. (CLRS) — *Introduction to Algorithms* | 4ª ed. | Cap. 6.4 | Heapsort |
| Cormen et al. (CLRS) — *Introduction to Algorithms* | 4ª ed. | Cap. 6.3 | Construcción del heap en O(n) |
| Goodrich, Tamassia & Goldwasser (GTG) — *Data Structures and Algorithms in Python* | 1ª ed. | Cap. 9.4 | Heapsort in-place |
| Miller & Ranum (M&R) — *Problem Solving with Algorithms and Data Structures Using Python* | 2011 | Cap. 6 | Binary heaps |
| Bhargava (Grok) — *Grokking Algorithms* | 2ª ed. | Cap. 2, 4 | Comparación de ordenamientos |

### Recursos gratuitos en línea

- 🌐 [VisuAlgo — Sorting](https://visualgo.net/en/sorting) — animación de Heapsort paso a paso.
- 🌐 [VisuAlgo — Binary Heap](https://visualgo.net/en/heap) — heapify y sortdown interactivos.
- 🎬 [Sorting Algorithms Visualized](https://www.toptal.com/developers/sorting-algorithms) — comparación sobre distintos perfiles de entrada.

### Práctica en Codeforces (soporta Python 3)

> 🔍 **Cómo filtrar:** ve a [codeforces.com/problemset](https://codeforces.com/problemset),
> escribe `sortings` o `data structures` en **Tags** y ajusta **Rating**.

| Rating | Nivel | Descripción |
|--------|-------|-------------|
| 800 | ⭐ | Aplicación directa |
| 1000 | ⭐⭐ | Requiere una pequeña adaptación |
| 1200 | ⭐⭐⭐ | Combina la idea con otra — desafío |

**Problemas recomendados:**

| # | Problema | Rating | Por qué es útil |
|---|----------|--------|-----------------|
| 1 | [158A — Next Round](https://codeforces.com/problemset/problem/158/A) | ⭐ 800 | Ordenar y consultar posiciones; aplicación directa |
| 2 | [456A — Laptops](https://codeforces.com/problemset/problem/456/A) | ⭐⭐ 1100 | Ordenar por una clave y detectar una inversión en la otra |
| 3 | [433B — Kuriyama Mirai's Stones](https://codeforces.com/problemset/problem/433/B) | ⭐⭐ 1200 | Ordenar una vez y responder muchas consultas: el costo del orden se amortiza |
| 4 | [451B — Sort the Array](https://codeforces.com/problemset/problem/451/B) | ⭐⭐⭐ 1300 | Razonar sobre qué segmento está desordenado sin ordenar todo |

⚠️ Los problemas 1 y 2 son el **mínimo esperado**. Los demás son desafío opcional.